In [ ]:
import sys, os, glob, shutil, subprocess
import torch

# Без видеокарты обучение бессмысленно: на процессоре 2M пар не досчитаются никогда.
# Падаем сразу, а не через часы занятой очереди.
if not torch.cuda.is_available():
    raise SystemExit("GPU не выделена — проверьте ускоритель в настройках ноутбука")
print("GPU:", torch.cuda.get_device_name(0))

# Kaggle распаковывает загруженное сам, поэтому точные пути заранее неизвестны.
found = glob.glob("/kaggle/input/**/llm_train.parquet", recursive=True)
if not found:
    raise SystemExit("Пакет данных не найден в /kaggle/input")
pack = os.path.dirname(found[0])
print("пакет:", pack, sorted(os.listdir(pack)))

# Модули лежат в датасете плоско, а запускать нужно как пакет `src`; дочерний процесс
# не наследует sys.path — поэтому собираем настоящий каталог пакета и задаём PYTHONPATH.
modules = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
if not modules:
    raise SystemExit("Код не найден в /kaggle/input")
os.makedirs("/kaggle/working/src", exist_ok=True)
for path in glob.glob(os.path.dirname(modules[0]) + "/*.py"):
    shutil.copy(path, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
print("модули:", sorted(os.listdir("/kaggle/working/src")))

subprocess.run([sys.executable, "-u", "-m", "src.train_ce_large",
                "--prepacked", pack, "--epochs", "1",
                "--batch-size", "256", "--max-length", "192",
                "--output", "/kaggle/working/ce_large"],
               check=True, cwd="/kaggle/working",
               env=dict(os.environ, PYTHONPATH="/kaggle/working"))
